<a href="https://colab.research.google.com/github/chamorrosandra214-rgb/BOTELLAS-DESCARTADAS/blob/main/BOTELLAS%20DESCARTADAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install simpy
import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1
PROB_DEFECTO = 0.15        # probabilidad de que una botella sea defectuosa

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)

        # Decisión: ¿aprobada o defectuosa?
        if random.random() < PROB_DEFECTO:
            descartadas.append(id_botella)
            print(f"❌ Botella {id_botella} descartada en {env.now:.1f}")
            return
        else:
            print(f"✅ Botella {id_botella} aprobada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"📦 Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque, descartadas):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []
descartadas = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque, descartadas))
env.run(until=500)  # simular 500 segundos

# Resumen final
print("\n--- RESULTADOS ---")
print(f"Botellas descartadas: {len(descartadas)}")
print(f"IDs descartados: {descartadas}")

Botella 1 moldeada en 37.0
Botella 2 moldeada en 37.2
Botella 3 moldeada en 62.5
Botella 4 moldeada en 66.7
Botella 5 moldeada en 86.9
Botella 1 enfriada en 97.0
Botella 2 enfriada en 97.2
❌ Botella 1 descartada en 104.1
❌ Botella 2 descartada en 112.2
Botella 3 enfriada en 122.5
Botella 4 enfriada en 126.7
✅ Botella 3 aprobada en 135.1
Botella 5 enfriada en 146.9
Botella 6 moldeada en 154.6
✅ Botella 4 aprobada en 178.9
Botella 7 moldeada en 189.5
✅ Botella 5 aprobada en 189.9
Botella 6 enfriada en 214.6
Botella 8 moldeada en 223.2
Botella 9 moldeada en 230.2
Botella 7 enfriada en 249.5
Botella 10 moldeada en 270.3
Botella 11 moldeada en 282.7
Botella 8 enfriada en 283.2
Botella 9 enfriada en 290.2
Botella 12 moldeada en 293.4
Botella 13 moldeada en 310.8
Botella 10 enfriada en 330.3
✅ Botella 6 aprobada en 338.2
✅ Botella 7 aprobada en 338.5
Botella 14 moldeada en 342.5
Botella 11 enfriada en 342.7
Botella 12 enfriada en 353.4
Botella 13 enfriada en 370.8
✅ Botella 8 aprobada en 378.